# 3주차. 나나의 기록장을 만들다

**부제:** BaseModel tool 입력 스키마로 SQLite에 저장: structured_requests, schedules, todos, reminders

## 0. 목표

이번 주에는 `SaveStructuredRequestInput(BaseModel)`로 저장 tool의 입력 스키마를 정의하고, 모델이 그 스키마에 맞춘 arguments로 SQLite 저장 tool을 호출하는 흐름을 만든다.

성취 기준:

- 프록시 토큰을 사용하는 LangChain tool call 흐름을 실행할 수 있다.
- `save_structured_request`의 arguments가 `BaseModel` 입력 스키마에 맞게 만들어졌는지 trace로 확인할 수 있다.
- `structured_requests`, `schedules`, `todos`, `reminders` table의 역할을 구분할 수 있다.

![save_db](imgs/save_db.jpg)

## 1. 준비

로컬 `Jupyter Notebook` 또는 `JupyterLab`에서 repo 루트를 기준으로 실행한다.

3주차도 프록시 서버를 통해 모델 API를 호출한다. `.env`의 `PROXY_TOKEN`, `PROXY_URL`, `OPENAI_MODEL`이 설정되어 있어야 한다.

In [1]:
from __future__ import annotations

import json
import os
import sqlite3
import sys
import uuid
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Literal

from dotenv import dotenv_values, load_dotenv
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / ".env").exists() or (candidate / ".env.example").exists():
            return candidate
    raise RuntimeError("repo root를 찾지 못했습니다. repo 안에서 노트북을 실행하세요.")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

# override=True를 사용해 노트북 커널에 남아 있던 이전 환경 변수보다 repo .env를 우선한다.
# 이 노트북에서 쓰는 프록시/모델 설정값은 repo 루트의 .env 파일에서만 읽는다.
ENV_PATH = REPO_ROOT / ".env"
load_dotenv(ENV_PATH, override=True)
ENV_VALUES = dotenv_values(ENV_PATH)


def required_env(name: str) -> str:
    value = (ENV_VALUES.get(name) or "").strip()
    if not value or value == "여기에 api key 입력":
        raise RuntimeError(f"repo 루트 .env 파일에 {name}을 설정한 뒤 다시 실행하세요.")
    return value


PROXY_TOKEN = required_env("PROXY_TOKEN")
PROXY_URL = required_env("PROXY_URL")
EMBEDDING_PROXY_URL = required_env("EMBEDDING_PROXY_URL")
OPENAI_MODEL = required_env("OPENAI_MODEL")
OPENAI_EMBEDDING_MODEL = required_env("OPENAI_EMBEDDING_MODEL")


def make_model(max_tokens: int = 700) -> ChatOpenAI:
    return ChatOpenAI(
        model=OPENAI_MODEL,
        api_key=PROXY_TOKEN,
        base_url=PROXY_URL,
        temperature=0,
        max_completion_tokens=max_tokens,
    )


def now_iso() -> str:
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def show_json(value: Any) -> None:
    print(json.dumps(value, ensure_ascii=False, indent=2, default=str))


def final_text(agent_result: dict[str, Any]) -> str:
    return agent_result["messages"][-1].content


def extract_tool_trace(agent_result: dict[str, Any]) -> list[dict[str, Any]]:
    trace: list[dict[str, Any]] = []
    for message in agent_result.get("messages", []):
        for call in getattr(message, "tool_calls", []) or []:
            trace.append({"event": "tool_call", "tool_name": call.get("name"), "arguments": call.get("args", {})})
        if getattr(message, "type", None) == "tool":
            trace.append({"event": "tool_result", "tool_name": getattr(message, "name", None), "content": message.content})
    return trace

DB_PATH = Path(os.getenv("KANAMATE_WEEK03_DB_PATH") or REPO_ROOT / "tmp" / "week03_structured_requests.sqlite3").resolve()
DB_PATH.parent.mkdir(parents=True, exist_ok=True)
show_json({"model": OPENAI_MODEL, "db_path": str(DB_PATH)})

{
  "model": "openai/gpt-4.1-mini",
  "db_path": "/Users/ssojux2/Working/kakao_clone_coding/tmp/week03_structured_requests.sqlite3"
}


## 2. 개념

- 모델은 자연어 요청을 자유 문장으로만 답하지 않고, `SaveStructuredRequestInput` 스키마에 맞는 tool arguments로 구조화한다.
- tool은 `BaseModel`로 검증되는 arguments를 받아 `structured_requests.payload_json`에 저장한다.
- `kind`가 일정/할 일/알림이면 같은 tool 안에서 해당 table에도 한 번 더 저장한다.

## 3. 기본 개념 실습

먼저 tool arguments로 사용할 `SaveStructuredRequestInput(BaseModel)`을 만든다. 이 모델이 `save_structured_request` tool의 입력 스키마다.

In [2]:
class SaveStructuredRequestInput(BaseModel):
    source_text: str = Field(description="사용자 원문 요청")
    kind: Literal["personal_schedule", "group_schedule", "todo", "reminder", "unknown"] = Field(description="사용자 요청 종류")
    title: str | None = Field(default=None, description="일정/할 일/알림 제목")
    date: str | None = Field(default=None, description="YYYY-MM-DD 날짜")
    start_time: str | None = Field(default=None, description="HH:MM 시작 시각")
    end_time: str | None = Field(default=None, description="HH:MM 종료 시각")
    members: list[str] = Field(default_factory=list, description="관련 참석자")
    priority: str | None = Field(default=None, description="low, medium, high 중 하나")
    reason: str | None = Field(default=None, description="분류 또는 추출 근거")


show_json({"tool_input_schema": list(SaveStructuredRequestInput.model_fields)})

{
  "tool_input_schema": [
    "source_text",
    "kind",
    "title",
    "date",
    "start_time",
    "end_time",
    "members",
    "priority",
    "reason"
  ]
}


## 4. 카나메이트 확장 예제

SQLite schema를 만들고, `SaveStructuredRequestInput`을 입력으로 받는 저장 tool 하나만 정의한다. tool 함수는 `**kwargs`로 입력을 받고, `BaseModel`로 검증한 payload를 `kind`에 맞는 table로 저장한다.

In [3]:
def connect_db() -> sqlite3.Connection:
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA foreign_keys = ON")
    return conn


def initialize_structured_db(reset: bool = False) -> None:
    with connect_db() as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS structured_requests (
                request_id TEXT PRIMARY KEY,
                source_text TEXT NOT NULL,
                kind TEXT NOT NULL,
                payload_json TEXT NOT NULL,
                created_at TEXT NOT NULL
            )
        """)
        conn.execute("""
            CREATE TABLE IF NOT EXISTS schedules (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                request_id TEXT NOT NULL,
                title TEXT NOT NULL,
                date TEXT,
                start_time TEXT,
                end_time TEXT,
                members_json TEXT NOT NULL,
                reason TEXT,
                schedule_type TEXT NOT NULL,
                FOREIGN KEY (request_id) REFERENCES structured_requests(request_id)
            )
        """)
        conn.execute("""
            CREATE TABLE IF NOT EXISTS todos (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                request_id TEXT NOT NULL,
                title TEXT NOT NULL,
                due_date TEXT,
                priority TEXT,
                reason TEXT,
                FOREIGN KEY (request_id) REFERENCES structured_requests(request_id)
            )
        """)
        conn.execute("""
            CREATE TABLE IF NOT EXISTS reminders (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                request_id TEXT NOT NULL,
                title TEXT NOT NULL,
                date TEXT,
                start_time TEXT,
                reason TEXT,
                FOREIGN KEY (request_id) REFERENCES structured_requests(request_id)
            )
        """)
        if reset:
            conn.execute("DELETE FROM reminders")
            conn.execute("DELETE FROM todos")
            conn.execute("DELETE FROM schedules")
            conn.execute("DELETE FROM structured_requests")
        conn.commit()


@tool("save_structured_request", description="구조화된 사용자 요청을 SQLite에 저장한다.", args_schema=SaveStructuredRequestInput)
def save_structured_request(**kwargs) -> str:
    """Save a structured request to SQLite."""
    payload = SaveStructuredRequestInput.model_validate(kwargs).model_dump()
    request_id = f"req-{uuid.uuid4().hex[:12]}"
    source_text = payload["source_text"]
    kind = payload["kind"]
    title = payload.get("title") or "제목 없음"
    date = payload.get("date")
    start_time = payload.get("start_time")
    end_time = payload.get("end_time")
    members = payload.get("members", [])
    priority = payload.get("priority")
    reason = payload.get("reason")
    payload["title"] = title
    payload["members"] = members
    normalized_table = None

    with connect_db() as conn:
        conn.execute(
            "INSERT INTO structured_requests (request_id, source_text, kind, payload_json, created_at) VALUES (?, ?, ?, ?, ?)",
            (request_id, source_text, kind, json.dumps(payload, ensure_ascii=False), now_iso()),
        )
        if kind in {"personal_schedule", "group_schedule"}:
            normalized_table = "schedules"
            conn.execute(
                """INSERT INTO schedules
                   (request_id, title, date, start_time, end_time, members_json, reason, schedule_type)
                   VALUES (?, ?, ?, ?, ?, ?, ?, ?)""",
                (request_id, title, date, start_time, end_time, json.dumps(members, ensure_ascii=False), reason, kind),
            )
        elif kind == "todo":
            normalized_table = "todos"
            conn.execute(
                "INSERT INTO todos (request_id, title, due_date, priority, reason) VALUES (?, ?, ?, ?, ?)",
                (request_id, title, date, priority, reason),
            )
        elif kind == "reminder":
            normalized_table = "reminders"
            conn.execute(
                "INSERT INTO reminders (request_id, title, date, start_time, reason) VALUES (?, ?, ?, ?, ?)",
                (request_id, title, date, start_time, reason),
            )
        conn.commit()

    return json.dumps({"ok": True, "request_id": request_id, "kind": kind, "normalized_table": normalized_table}, ensure_ascii=False)


structured_save_agent = create_agent(
    model=make_model(700),
    tools=[save_structured_request],
    system_prompt=(
        "너는 카나메이트 요청 저장 에이전트다. 오늘은 2026-05-15이다. "
        "사용자 요청을 personal_schedule, group_schedule, todo, reminder, unknown 중 하나로 분류한다. "
        "source_text에는 사용자 원문을 그대로 넣고, 필요한 필드를 채워 save_structured_request 도구를 한 번 호출한다. "
        "상대 날짜는 오늘 기준으로 YYYY-MM-DD로 바꾼다. 도구 호출 뒤에는 짧게 답한다."
    ),
)


initialize_structured_db(reset=True)
show_json({"registered_tool": save_structured_request.name})

{
  "registered_tool": "save_structured_request"
}


## 5. 확장 예제 실행

요청 하나를 실행한 뒤 `tool_call.arguments`가 `SaveStructuredRequestInput` 형태인지, 그리고 SQLite row가 저장됐는지 확인한다.

In [4]:
def fetch_all(table: str) -> list[dict[str, Any]]:
    with connect_db() as conn:
        rows = conn.execute(f"SELECT * FROM {table} ORDER BY rowid").fetchall()
    return [dict(row) for row in rows]


request_text = "민수 지아랑 다음 주 화요일 3시에 회의 잡아줘. 중요한 일정이야."
result = structured_save_agent.invoke({"messages": [{"role": "user", "content": request_text}]})
trace = extract_tool_trace(result)

saved_rows = {
    "structured_requests": fetch_all("structured_requests"),
    "schedules": fetch_all("schedules"),
    "todos": fetch_all("todos"),
    "reminders": fetch_all("reminders"),
}

show_json({"answer": final_text(result), "tool_trace": trace, "saved_rows": saved_rows})

save_calls = [event for event in trace if event["event"] == "tool_call" and event["tool_name"] == "save_structured_request"]
assert len(save_calls) == 1
assert save_calls[0]["arguments"]["kind"] == "group_schedule"
assert len(saved_rows["structured_requests"]) == 1
assert len(saved_rows["schedules"]) == 1
assert len(saved_rows["todos"]) == 0
assert len(saved_rows["reminders"]) == 0

{
  "answer": "다음 주 화요일(5월 19일) 오후 3시에 민수, 지아와 중요한 회의 일정을 잡았습니다.",
  "tool_trace": [
    {
      "event": "tool_call",
      "tool_name": "save_structured_request",
      "arguments": {
        "source_text": "민수 지아랑 다음 주 화요일 3시에 회의 잡아줘. 중요한 일정이야.",
        "kind": "group_schedule",
        "title": "회의",
        "date": "2026-05-19",
        "start_time": "15:00",
        "members": [
          "민수",
          "지아"
        ],
        "priority": "high",
        "reason": "다음 주 화요일 3시에 민수, 지아와 회의 요청, 중요한 일정"
      }
    },
    {
      "event": "tool_result",
      "tool_name": "save_structured_request",
      "content": "{\"ok\": true, \"request_id\": \"req-ae54dfd355c1\", \"kind\": \"group_schedule\", \"normalized_table\": \"schedules\"}"
    }
  ],
  "saved_rows": {
    "structured_requests": [
      {
        "request_id": "req-ae54dfd355c1",
        "source_text": "민수 지아랑 다음 주 화요일 3시에 회의 잡아줘. 중요한 일정이야.",
        "kind": "group_schedule",
        "payload_json": "{\"source_text\": \

## 6. 회고

확인 질문:

1. `SaveStructuredRequestInput(BaseModel)`이 tool 입력에서 맡는 역할은 무엇인가?
2. `tool_call.arguments`에서 모델이 구조화한 값은 무엇인가?
3. 원본 arguments를 `structured_requests.payload_json`에 남기는 이유는 무엇인가?

작은 응용 과제: `todo`, `reminder`, `unknown` 요청으로 바꿔 실행하고 어느 table에 저장되는지 비교한다.